# Attention maps of the 12-class classifier (`src/knee_cls`)

Visualises where the trained classifier looks, for each of the 12 findings, on **one study**, and turns the attention into rough segmentations.

- **Model:** 2.5D attention network trained with `src/main_cls.py`. A 2D CNN (ConvNeXt) sees one *frame* at a time (3 adjacent slices as channels, 224 px). Per class it has an **in-plane attention map** (softmax over the stride-8 grid), then an **attention over frames** of the series (z), then an **attention over the series** of the study. The product of the first two is a 3D attention volume per class and per series, summing to 1.
- **Labels:** trained on the Claude report labels only. No pixel-level supervision: the maps show *where the evidence for a class was read*, which is not always the anatomical structure the class is named after.
- **Dense frames:** for visualisation a frame is centred on every slice of the series (training uses 6 strided frames), so every slice gets a map, except the outer 10% at each end which training also leaves out. The z softmax is then spread over more frames than in training; it is the *relative* z profile that matters.
- **Comparison with nnU-Net:** the default study is the one `02_segmentation_nnunet.ipynb` picked, so where its masks exist the attention is cross-checked against bones, cartilage and menisci.
- Export for many studies at once: `python main_attention.py --run_dir ... --studies gt` (one `.npz` per series, see its docstring).

In [ ]:
import os, sys, glob
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import SimpleITK as sitk
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = Path('..').resolve()
SRC = ROOT / 'src'
sys.path.insert(0, str(SRC))
os.chdir(SRC)                      # params.py paths are relative to src/
from main_attention import StudyAttention, to_original
from params import CLASSES

RUN_DIR = None          # None: most recent run under logs/cls_run/ with a ckpt_final.pt
STUDY_UID = None        # None: same automatic choice as the segmentation notebooks
FOCUS_CLASS = 'ACL'     # class shown slice by slice in section 5
SHOW_CLASSES = CLASSES  # classes in the per-class figures
MASS = 0.5              # attention -> mask: smallest region holding this fraction of the attention
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if RUN_DIR is None:
    RUN_DIR = os.path.dirname(max(glob.glob(str(ROOT / 'logs' / 'cls_run' / '*' / '*' / 'ckpt_final.pt')),
                                  key=os.path.getmtime))
print('run', RUN_DIR)

## 1. Model, study and predictions

In [ ]:
sa = StudyAttention(RUN_DIR, device=DEVICE)
idx = pd.read_csv(ROOT / 'output' / 'series_index.csv').set_index('SeriesInstanceUID')

if STUDY_UID is None:   # labeled study with a fluid-sensitive series in all 3 views, as in 02/03
    train = pd.read_csv(ROOT / 'input' / 'train.csv', usecols=['StudyInstanceUID', 'ACL'])
    fluid = idx[idx.Fluid_Sensitive == 1]
    complete = fluid.groupby('StudyInstanceUID').Anatomical_Plane.nunique()
    STUDY_UID = sorted(set(complete[complete == 3].index) & set(train.dropna().StudyInstanceUID))[0]

probs, recs = sa(STUDY_UID, dense=True)
for r in recs:
    r['plane'] = idx.loc[r['series_uid'], 'Anatomical_Plane']
    r['fluid'] = int(idx.loc[r['series_uid'], 'Fluid_Sensitive'])
    r['name'] = f"{r['plane'][:3]} {'fluid' if r['fluid'] else 'T1/PD'} ..{r['series_uid'][-6:]}"
    r['vol'] = np.load(ROOT / idx.loc[r['series_uid'], 'array_path'], mmap_mode='r')

st = sa.studies.set_index('StudyInstanceUID').loc[STUDY_UID]
print('study', STUDY_UID, '|', len(recs), 'series')
pd.DataFrame({'predicted': probs.round(3),
              'Claude label': [st[f'target_{c}'] for c in CLASSES],
              'radiologist': [st[f'gt_{c}'] for c in CLASSES]}, index=CLASSES)

## 2. Which series each class reads

Series attention: for each class, the weights over the study's series (columns sum to 1). Uniform would be 1 / number of series.

In [ ]:
W = np.stack([r['series_attn'] for r in recs])           # S K
fig, ax = plt.subplots(figsize=(13, 0.5 * len(recs) + 1.5))
ax.imshow(W, cmap='viridis', vmin=0, aspect='auto')
ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=35, ha='right')
ax.set_yticks(range(len(recs)), [r['name'] for r in recs])
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        ax.text(j, i, f'{W[i, j]:.2f}', ha='center', va='center', fontsize=8,
                color='w' if W[i, j] < W.max() * 0.6 else 'k')
ax.set_title(f'series attention (uniform = {1 / len(recs):.2f})')
plt.tight_layout(); plt.show()

## 3. Which slices each class reads

z attention per series, scaled by the number of frames so that **1 = uniform**. Peaks mark the slices a class relies on; flat curves mean the series is read as a whole (or not used, see section 2).

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 10), sharey=False)
for ax, c in zip(axes.ravel(), CLASSES):
    k = CLASSES.index(c)
    for r in recs:
        F = r['z_attn'].shape[1]
        ax.plot(r['slices'], r['z_attn'][k] * F, lw=1 + 3 * r['series_attn'][k], label=r['name'])
    ax.axhline(1, c='gray', ls=':', lw=0.8)
    ax.set_title(f'{c}  p={probs[k]:.2f}', fontsize=10); ax.set_xlabel('slice')
axes[0, 0].legend(fontsize=7)
plt.suptitle('z attention x n_frames (line width = series attention)'); plt.tight_layout(); plt.show()

## 4. In-plane attention, per class

For each class: the series it weights most, on its 3 highest-z-attention slices, with the in-plane map (normalised per slice) overlaid.

In [ ]:
def norm_img(img):
    lo, hi = np.percentile(img, [1, 99.5])
    return np.clip((np.asarray(img, np.float32) - lo) / max(hi - lo, 1e-6), 0, 1)


def best_series(k):
    return max(recs, key=lambda r: r['series_attn'][k])


N_TOP = 3
fig, axes = plt.subplots(len(SHOW_CLASSES), N_TOP, figsize=(3.3 * N_TOP, 3.4 * len(SHOW_CLASSES)), squeeze=False)
for row, c in zip(axes, SHOW_CLASSES):
    k = CLASSES.index(c)
    r = best_series(k)
    for a, f in zip(row, sorted(np.argsort(-r['z_attn'][k])[:N_TOP])):
        att = to_original(r['xy_attn'][k, f].astype(np.float32), r['rows'], r['cols'])
        a.imshow(norm_img(r['vol'][r['slices'][f]]), cmap='gray')
        a.imshow(att / att.max(), cmap='jet', alpha=0.35, vmin=0, vmax=1)
        a.set_title(f"{c} p={probs[k]:.2f} | {r['name']}\nslice {r['slices'][f]}  z x F={r['z_attn'][k, f] * r['z_attn'].shape[1]:.1f}", fontsize=8)
        a.axis('off')
plt.tight_layout(); plt.show()

## 5. One class through its series

Every slice of the series `FOCUS_CLASS` weights most, with the **3D** attention (z x in-plane) overlaid on a common scale: slices the class ignores fade out.

In [ ]:
k = CLASSES.index(FOCUS_CLASS)
r = best_series(k)
A = to_original(r['attn'][k], r['rows'], r['cols'])          # F rows cols
vmax = A.max()
ncols = 8
nrows = int(np.ceil(len(r['slices']) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(2.4 * ncols, 2.5 * nrows), squeeze=False)
for a in axes.ravel(): a.axis('off')
for a, f in zip(axes.ravel(), range(len(r['slices']))):
    a.imshow(norm_img(r['vol'][r['slices'][f]]), cmap='gray')
    a.imshow(A[f] / vmax, cmap='jet', alpha=np.clip(A[f] / vmax, 0, 1) * 0.8, vmin=0, vmax=1)
    a.set_title(f"slice {r['slices'][f]}", fontsize=8)
plt.suptitle(f"{FOCUS_CLASS} (p={probs[k]:.2f}) on {r['name']}: 3D attention, common scale", y=1.0)
plt.tight_layout(); plt.show()

## 6. From attention to masks

A mask per class and slice: the smallest set of pixels holding `MASS` of that slice's **in-plane** attention. One panel per class, on the series the class weights most and the slice where its z attention peaks: contour on top of the faint attention map.

In [ ]:
def attention_mask(a, mass=MASS):
    # Smallest pixel set holding `mass` of the attention map a (any shape).
    flat = a.ravel() / a.sum()
    order = np.argsort(-flat)
    keep = np.zeros(flat.size, bool)
    keep[order[:np.searchsorted(np.cumsum(flat[order]), mass) + 1]] = True
    return keep.reshape(a.shape)


def class_view(k):
    # (series record, frame index, upsampled in-plane map) where class k looks most.
    r = best_series(k)
    f = int(np.argmax(r['z_attn'][k]))
    return r, f, to_original(r['xy_attn'][k, f].astype(np.float32), r['rows'], r['cols'])


ncols = 4
nrows = int(np.ceil(len(SHOW_CLASSES) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 4.4 * nrows), squeeze=False)
for a in axes.ravel(): a.axis('off')
for a, c in zip(axes.ravel(), SHOW_CLASSES):
    k = CLASSES.index(c)
    r, f, att = class_view(k)
    a.imshow(norm_img(r['vol'][r['slices'][f]]), cmap='gray')
    a.imshow(att / att.max(), cmap='jet', alpha=0.2)
    a.contour(attention_mask(att), levels=[0.5], colors=['red'], linewidths=1.5)
    a.set_title(f"{c} p={probs[k]:.2f}\n{r['name']} slice {r['slices'][f]}", fontsize=9)
plt.suptitle(f'{int(MASS * 100)}% in-plane attention masks', y=1.0); plt.tight_layout(); plt.show()

## 7. Cross-check with the nnU-Net masks

Where `02_segmentation_nnunet.ipynb` produced a segmentation for a series of this study, the table gives, for each class, the **enrichment** of its 3D attention on each structure: share of the attention on the structure / share of the volume the structure occupies (masks dilated by 3 px, since the attention grid is 8x coarser than the image). 1 = no preference, >> 1 = the class reads that structure. Structures nnU-Net barely found in this series (< 0.2% of the volume) are left out: on these 2D, thick-slice series it often misses cartilage and menisci. There is no ACL / MCL mask: those can only be judged visually.

In [ ]:
from scipy import ndimage as ndi

NN_LABELS = {1: 'patellar cart.', 2: 'femoral cart.', 3: 'med. tibial cart.', 4: 'lat. tibial cart.',
             5: 'med. meniscus', 6: 'lat. meniscus', 7: 'femur', 8: 'tibia', 9: 'patella'}
nn_dir = ROOT / 'output' / 'segmentation' / 'nnunet'
found = []
for r in recs:
    case = f"{r['plane'].lower()}_{r['series_uid'][-8:]}"
    p = nn_dir / case / 'cascade' / f'{case}.nii.gz'
    if p.exists():
        r['nn'] = sitk.GetArrayFromImage(sitk.ReadImage(str(p))).astype(np.uint8)
        found.append(r)
print('series with an nnU-Net mask:', [r['name'] for r in found] or 'none (run 02_segmentation_nnunet.ipynb on this study)')

tables = {}
for r in found:
    seg = r['nn'][r['slices']]                                      # F rows cols, same slices as the attention
    masks = {name: ndi.binary_dilation(seg == lab, iterations=3, structure=np.ones((1, 3, 3)))
             for lab, name in NN_LABELS.items()}
    masks = {name: m for name, m in masks.items() if m.mean() >= 0.002}
    rows = {}
    for c in SHOW_CLASSES:
        k = CLASSES.index(c)
        A = to_original(r['attn'][k], r['rows'], r['cols']); A = A / A.sum()
        rows[c] = {name: A[m].sum() / m.mean() for name, m in masks.items()}
    t = pd.DataFrame(rows).T
    t.loc['(area %)'] = [100 * m.mean() for m in masks.values()]
    tables[r['name']] = t
for name, t in tables.items():
    print(name, '-- enrichment over chance (last row: structure area in %)')
    display(t.style.format('{:.1f}').background_gradient(axis=None, cmap='Blues', subset=pd.IndexSlice[t.index[:-1], :]))

In [ ]:
# Menisci: nnU-Net mask vs. attention mask, on the slices where each meniscus class looks most.
for r in found:
    if r['plane'] != 'Sagittal':
        continue
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
    for (c, lab), pair in zip([('Medial Meniscus', 5), ('Lateral Meniscus', 6)], [axes[:2], axes[2:]]):
        k = CLASSES.index(c)
        f0 = int(np.argmax(r['z_attn'][k]))
        m = attention_mask(to_original(r['xy_attn'][k, f0].astype(np.float32), r['rows'], r['cols']))
        img = norm_img(r['vol'][r['slices'][f0]])
        nnm = r['nn'][r['slices'][f0]] == lab
        for a, title in zip(pair, ['image', 'nnU-Net (green) vs attention mask (red)']):
            a.imshow(img, cmap='gray'); a.axis('off')
            a.set_title(f"{c}, slice {r['slices'][f0]}: {title}", fontsize=9)
        if nnm.any():
            pair[1].contour(nnm, levels=[0.5], colors=['lime'], linewidths=1.2)
        pair[1].contour(m, levels=[0.5], colors=['red'], linewidths=1.2)
    plt.tight_layout(); plt.show()

## 8. Attention heads

Runs trained with `--xy_heads > 1` have several in-plane maps per class, each pooling its own channel group. Distinct evidence can then take distinct heads, e.g. for ACL one head on the ligament and another on pivot-shift contusions. Shown for `FOCUS_CLASS` on its top slice; skipped for single-head runs.

In [ ]:
k = CLASSES.index(FOCUS_CLASS)
r = best_series(k)
if 'xy_heads' not in r:
    print('single-head run: nothing to show')
else:
    H = r['xy_heads'].shape[1]
    f0 = int(np.argmax(r['z_attn'][k]))
    fig, axes = plt.subplots(1, H + 1, figsize=(3.6 * (H + 1), 3.8))
    img = norm_img(r['vol'][r['slices'][f0]])
    axes[0].imshow(img, cmap='gray'); axes[0].set_title(f"{r['name']} slice {r['slices'][f0]}", fontsize=9)
    for h in range(H):
        att = to_original(r['xy_heads'][k, h, f0].astype(np.float32), r['rows'], r['cols'])
        axes[h + 1].imshow(img, cmap='gray')
        axes[h + 1].imshow(att / att.max(), cmap='jet', alpha=0.4)
        axes[h + 1].set_title(f'{FOCUS_CLASS} head {h}', fontsize=9)
    for a in axes: a.axis('off')
    plt.tight_layout(); plt.show()

### How to read this

- **What the maps are.** Attention is where evidence for a class was *read*, learned from study-level report labels only. Classes tied to one structure (menisci, effusion in the suprapatellar recess, Baker's cyst behind the knee, compartment OA) tend to land on it. Classes with strong secondary signs may not: in an ACL-torn knee the ACL map can sit on the posterior tibial plateau, where pivot-shift contusions are, rather than on the ligament.
- **Negative studies** are informative too: the in-plane softmax always puts its mass somewhere, usually where the class *would* be found, so maps from normal knees localise the anatomy the class is about.
- **Resolution.** The maps live on a stride-8 grid of the 224 px input (28 x 28), upsampled here: they give location and extent, not boundaries. For boundaries, use them as prompts (box / point) for a segmentation model, as `03_segmentation_samri.ipynb` does with nnU-Net masks.
- **Masks** in section 6 are per slice and depend on `MASS`; sharper attention gives smaller masks at the same `MASS`. For a 3D mask, threshold `attn` (z x in-plane) per slice the same way, or keep slices whose `z_logits` stand out.